# Download the dataset from SatNOGS API and their S3 bucket for waterfalls

You must set your SatNOGS API token in .env in order to avoid rate limiting

In [1]:
import os
from dotenv import load_dotenv
import requests
import time

load_dotenv()

API_TOKEN = os.getenv("API_TOKEN")
header = {'Authorization': f'Token {API_TOKEN}'}
signalurl = "https://network.satnogs.org/api/observations/?waterfall_status=1&format=json"
nosignalurl = "https://network.satnogs.org/api/observations/?waterfall_status=0&format=json"
NUM_PAGES = 800*2

In [ ]:
os.makedirs("data", exist_ok=True)
os.makedirs("data/with_signal", exist_ok=True)
os.makedirs("data/without_signal", exist_ok=True)

In [ ]:
link = signalurl

def download_waterfalls(url, folder, num_pages):
    link = url
    for i in range(num_pages - 1):
        response = requests.get(link, headers=header)
        if response.status_code == 429:
            print("Rate limit exceeded. Waiting...")
            time.sleep(60)
            response = requests.get(link, headers=header)

        observation_ids = [k['id'] for k in response.json()]
        waterfall_urls = [k['waterfall'] for k in response.json()]
        transmitter_modes = [k['transmitter_mode'] or 'UNKNWN' for k in response.json()]
        print(f"Page {i+1}: Downloading {len(waterfall_urls)} waterfalls")
        for obs_id, waterfall_url, transmitter_mode in zip(observation_ids, waterfall_urls, transmitter_modes):
            waterfall_data = requests.get(waterfall_url).content
            with open(f"{folder}/{obs_id}_{transmitter_mode}.png", "wb") as f:
                f.write(waterfall_data)
        link = response.links.get('next', {}).get('url', None)
        if link is None:
            break

In [ ]:

download_waterfalls(signalurl, "data/with_signal", NUM_PAGES)

In [ ]:
download_waterfalls(nosignalurl, "data/without_signal", NUM_PAGES)

## Save the files somewhere and move on to preprocess.ipynb